# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

# Stage 1 — Tier 2 metrics: calibration

Reads the scored parquets written by `stage_1.ipynb` (with PriDe debiasing)
and produces calibration diagnostics specified in §2.2 of the report:

- **Top-label ECE** with 15 equal-width bins (the standard MLcommunity definition).
- **Classwise ECE**, computed per class on $P(Y = c \mid X)$ and averaged
  across classes. More sensitive to minority-group calibration than
  top-label.
- **Reliability diagrams**, one per (target, kind), with both top-label
  and classwise overlays.

ECE is reported as a secondary diagnostic; Brier (Tier 1) is the
primary calibration-sensitive scoring rule. ECE is binning-sensitive
and noisy in sparse bins, so we additionally report:

- **Adaptive (equal-mass) ECE**, which uses 15 quantile bins instead of
  equal-width. Less affected by sparse high-confidence bins.
- **Bin populations**, so the reader can see which bins are statistically
  meaningful.

All metrics are computed on the **debiased** probabilities (canonical),
with raw shown in an appendix table.

**Section overview**

| §  | Output                                                        |
|----|---------------------------------------------------------------|
| 1  | Setup: load all 7 scored parquets                             |
| 2  | ECE primitives (top-label, classwise, equal-width / equal-mass)|
| 3  | Reliability-diagram data builder                              |
| 4  | Per-target ECE table (raw + debiased side-by-side)            |
| 5  | Reliability-diagram figures (per target, top-label)           |
| 6  | Classwise reliability for the most-imbalanced target          |
| 7  | Cross-target summary                                          |


## 1. Setup

Same loader pattern as the Tier 1 notebook. Loads scored parquets and
detects PriDe schema vs legacy.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Model under analysis -------------------------------------------------
# MODEL_NAME = "mistralai/Ministral-3-14B-Instruct-2512"
#MODEL_NAME = "Qwen/Qwen3-30B-A3B-Instruct-2507"
#MODEL_NAME = "Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_balanced_05_11"
#MODEL_NAME = "Qwen2.5-14B-Instruct_baseline_test_balanced_05_11"

# MODEL_NAME = "Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_balanced_05_11"
#MODEL_NAME = "Qwen2.5-14B-Instruct_baseline_test_balanced_05_11"

# MODEL_NAME = "Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_raw_05_12_lauren"
#MODEL_NAME = "Qwen2.5-14B-Instruct-FT_balanced_canonical_step07_bf16_test_raw_05_12"
MODEL_NAME = "Qwen2.5-14B-Instruct_baseline_test_raw_05_12"

# --- Display ---
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

# --- ECE config ---
N_BINS_EQW = 15           # equal-width bins (the §2.2 spec)
N_BINS_EQM = 15           # equal-mass quantile bins (sensitivity check)
GLOBAL_SEED = 42

# --- I/O ---
SCORED_DIR  = Path(
    f"/content/drive/MyDrive/NLP_project/data/"
    f"stage_1_output_{MODEL_NAME}/"
)
METRICS_DIR = Path(
    f"/content/drive/MyDrive/NLP_project/data/"
    f"stage_1_output_{MODEL_NAME}/phase1_metrics/"
)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "SEX",
    "AGE_BINS",
    "RACE_ETHNICITY",
    "EDUCATION_LEVEL",
    "INCTOT_BINS",
    "HEALTH_INSURANCE",
    "ENGLISH_STATUS",
]


In [ ]:
def detect_prob_kinds(scored: pd.DataFrame, classes: list) -> list:
    has_raw = all(f"prob_raw_{c}" in scored.columns for c in classes)
    has_deb = all(f"prob_deb_{c}" in scored.columns for c in classes)
    if has_raw and has_deb:
        return ["raw", "deb"]
    return ["raw"]


def load_target(target: str) -> dict:
    pq_path  = SCORED_DIR / f"phase1_scored_{target}.parquet"
    meta_path = SCORED_DIR / f"phase1_scored_{target}.meta.json"
    if not pq_path.exists():
        raise FileNotFoundError(f"Missing scored parquet for {target}: {pq_path}")
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing metadata for {target}: {meta_path}")
    scored = pd.read_parquet(pq_path)
    with open(meta_path) as f:
        meta = json.load(f)
    classes = meta["class_labels"]
    prob_kinds = detect_prob_kinds(scored, classes)
    needs_legacy = (prob_kinds == ["raw"]
                    and not all(f"prob_raw_{c}" in scored.columns for c in classes))
    if needs_legacy:
        for c in classes:
            scored[f"prob_raw_{c}"] = scored[f"prob_{c}"]
    return {
        "scored":        scored,
        "meta":          meta,
        "classes":       classes,
        "prob_kinds":    prob_kinds,
        "applied_pride": meta.get("apply_pride", False),
    }


def prob_cols(classes: list, kind: str = "deb") -> list:
    return [f"prob_{kind}_{c}" for c in classes]


runs = {}
for t in TARGETS:
    try:
        runs[t] = load_target(t)
        n = len(runs[t]["scored"])
        k = len(runs[t]["classes"])
        kinds = "+".join(runs[t]["prob_kinds"])
        pride = "PriDe" if runs[t]["applied_pride"] else "raw-only"
        print(f"  {t:<20} loaded: {n:,} rows, {k} classes, "
              f"kinds=[{kinds}] ({pride})")
    except FileNotFoundError as e:
        warnings.warn(f"Skipping {t}: {e}")

print(f"\nLoaded {len(runs)} / {len(TARGETS)} targets.")
ALL_HAVE_DEB = all("deb" in r["prob_kinds"] for r in runs.values())
DEFAULT_KIND = "deb" if ALL_HAVE_DEB else "raw"
print(f"Default kind for headline: {DEFAULT_KIND!r}")


## 2. ECE primitives

The function `binned_calibration` is the core primitive: given a
confidence vector $p \in [0, 1]^N$ and a correctness vector
$y \in \{0, 1\}^N$, it returns per-bin (mean confidence, mean
accuracy, bin count) plus the weighted ECE.

**Top-label ECE.** $p$ = top class probability per row, $y$ = 1 if
argmax matches truth.

**Classwise ECE.** For each class $c$, $p$ = $P(Y = c \mid X)$,
$y$ = $\mathbb{1}[Y = c]$. Compute ECE on this binary problem, then
average across classes. The "macro" version (unweighted average) is
what we use; weighted-by-class-frequency would just recover top-label.

Two binning schemes:

- **Equal-width**: 15 bins of width $1/15$ over [0, 1]. Standard
  ML-community choice. Sparse in high-confidence regions for
  middling-accuracy models.
- **Equal-mass** (quantile-based): 15 bins each containing ~$N/15$
  rows. More stable in regions where the model is rarely confident,
  but bin widths vary.

We report both and treat large discrepancies as a sign that the
single-number ECE is unreliable (then defer to Brier).


In [ ]:
def equal_width_edges(n_bins: int) -> np.ndarray:
    """Edges for equal-width binning over [0, 1], inclusive at 1."""
    return np.linspace(0.0, 1.0, n_bins + 1)


def equal_mass_edges(p: np.ndarray, n_bins: int) -> np.ndarray:
    """Edges for quantile binning, with safe handling of duplicates.

    np.quantile with linspace probability levels can give duplicate edges
    when the data is concentrated. We monotonize by appending tiny epsilons
    so np.digitize still produces n_bins non-empty bins where possible.
    """
    qs = np.linspace(0.0, 1.0, n_bins + 1)
    edges = np.quantile(p, qs)
    # Force the endpoints exactly at [0, 1] (otherwise one row can fall
    # outside the bins due to floating-point roundoff).
    edges[0]  = 0.0
    edges[-1] = 1.0 + 1e-9
    # Monotonize: if any consecutive edges are equal, bump the right one
    # by a tiny epsilon so digitize doesn't collapse them.
    for i in range(1, len(edges)):
        if edges[i] <= edges[i - 1]:
            edges[i] = edges[i - 1] + 1e-12
    return edges


def binned_calibration(p: np.ndarray, y: np.ndarray,
                        edges: np.ndarray) -> dict:
    """Compute per-bin (conf, acc, count) and weighted ECE.

    Args:
      p     : confidence values, shape (N,), in [0, 1].
      y     : binary correctness, shape (N,), in {0, 1}.
      edges : bin edges, shape (n_bins + 1,).

    Returns dict with:
      bin_conf  : (n_bins,) mean confidence per bin (NaN if empty).
      bin_acc   : (n_bins,) mean accuracy per bin (NaN if empty).
      bin_count : (n_bins,) number of rows per bin.
      bin_lo    : (n_bins,) left edge per bin.
      bin_hi    : (n_bins,) right edge per bin.
      ece       : weighted average |acc - conf|.
    """
    n_bins = len(edges) - 1
    # np.digitize returns indices in [1, n_bins+1]. Use right=False so the
    # rightmost bin includes 1.0 (we already nudged the upper edge above 1).
    idx = np.digitize(p, edges, right=False) - 1
    idx = np.clip(idx, 0, n_bins - 1)

    bin_conf  = np.full(n_bins, np.nan)
    bin_acc   = np.full(n_bins, np.nan)
    bin_count = np.zeros(n_bins, dtype=np.int64)

    for b in range(n_bins):
        mask = (idx == b)
        bin_count[b] = int(mask.sum())
        if bin_count[b] > 0:
            bin_conf[b] = float(p[mask].mean())
            bin_acc[b]  = float(y[mask].mean())

    # Weighted ECE: only bins with count > 0 contribute.
    nonempty = bin_count > 0
    if nonempty.any() and len(p) > 0:
        weights = bin_count[nonempty] / bin_count.sum()
        ece = float((weights * np.abs(bin_acc[nonempty]
                                       - bin_conf[nonempty])).sum())
    else:
        ece = float("nan")

    return {
        "bin_conf":  bin_conf,
        "bin_acc":   bin_acc,
        "bin_count": bin_count,
        "bin_lo":    edges[:-1],
        "bin_hi":    edges[1:],
        "ece":       ece,
    }


def top_label_calibration(P: np.ndarray, y: np.ndarray,
                           n_bins: int = N_BINS_EQW,
                           binning: str = "equal_width") -> dict:
    """Top-label ECE on multi-class probabilities P (N, n_Y) with truth y."""
    top_conf = P.max(axis=1)
    top_pred = P.argmax(axis=1)
    correct  = (top_pred == y).astype(np.float64)

    if binning == "equal_width":
        edges = equal_width_edges(n_bins)
    elif binning == "equal_mass":
        edges = equal_mass_edges(top_conf, n_bins)
    else:
        raise ValueError(f"unknown binning: {binning}")
    return binned_calibration(top_conf, correct, edges)


def classwise_calibration(P: np.ndarray, y: np.ndarray, n_classes: int,
                           n_bins: int = N_BINS_EQW,
                           binning: str = "equal_width") -> dict:
    """Classwise ECE: per-class binary calibration, averaged across classes.

    Returns dict with:
      per_class_ece : (n_classes,) ECE for each class
      per_class_n   : (n_classes,) number of positive rows per class
      mean_ece      : unweighted average across classes
      details       : list of per-class binned_calibration outputs
    """
    per_class_ece = np.full(n_classes, np.nan)
    per_class_n   = np.zeros(n_classes, dtype=np.int64)
    details = []

    for c in range(n_classes):
        p_c     = P[:, c]
        y_c     = (y == c).astype(np.float64)
        if binning == "equal_width":
            edges = equal_width_edges(n_bins)
        else:
            edges = equal_mass_edges(p_c, n_bins)
        cal = binned_calibration(p_c, y_c, edges)
        per_class_ece[c] = cal["ece"]
        per_class_n[c]   = int(y_c.sum())
        details.append(cal)

    mean_ece = float(np.nanmean(per_class_ece))
    return {
        "per_class_ece": per_class_ece,
        "per_class_n":   per_class_n,
        "mean_ece":      mean_ece,
        "details":       details,
    }


## 3. Compute calibration diagnostics for every target × kind

For each (target, kind) we compute four ECE numbers:

- top-label, equal-width
- top-label, equal-mass
- classwise, equal-width
- classwise, equal-mass

Plus the per-bin reliability data needed for the figures.


In [ ]:
def assemble_prob_matrix(scored, classes, kind):
    return scored[prob_cols(classes, kind)].to_numpy()


def true_indices(scored, target_col, classes):
    cls_to_idx = {c: i for i, c in enumerate(classes)}
    return scored[f"true_{target_col}"].map(cls_to_idx).to_numpy()


# Compute everything up front. Cheap (no bootstrap; ECE is just histograms).
calib_data = {}   # (target, kind) -> dict
for t, run in runs.items():
    P_kinds = {kind: assemble_prob_matrix(run["scored"], run["classes"], kind)
               for kind in run["prob_kinds"]}
    y = true_indices(run["scored"], t, run["classes"])
    n_classes = len(run["classes"])

    for kind, P in P_kinds.items():
        tl_eqw = top_label_calibration(P, y, N_BINS_EQW, "equal_width")
        tl_eqm = top_label_calibration(P, y, N_BINS_EQM, "equal_mass")
        cw_eqw = classwise_calibration(P, y, n_classes, N_BINS_EQW, "equal_width")
        cw_eqm = classwise_calibration(P, y, n_classes, N_BINS_EQM, "equal_mass")

        calib_data[(t, kind)] = {
            "top_label_eqw": tl_eqw,
            "top_label_eqm": tl_eqm,
            "classwise_eqw": cw_eqw,
            "classwise_eqm": cw_eqm,
            "n_rows":        len(y),
            "n_classes":     n_classes,
        }

print(f"Computed calibration for {len(calib_data)} (target, kind) pairs.")


## 4. Per-target ECE table

One row per (target, kind) with all four ECE numbers plus a "stable?"
flag indicating whether equal-width and equal-mass agree to within 0.02.
Large disagreement → bin sensitivity is biting and the scalar ECE
shouldn't be taken too seriously (use the reliability diagrams + Brier).


In [ ]:
ece_rows = []
for t in runs:
    for kind in runs[t]["prob_kinds"]:
        d = calib_data[(t, kind)]
        tl_w = d["top_label_eqw"]["ece"]
        tl_m = d["top_label_eqm"]["ece"]
        cw_w = d["classwise_eqw"]["mean_ece"]
        cw_m = d["classwise_eqm"]["mean_ece"]
        stable = (abs(tl_w - tl_m) < 0.02) and (abs(cw_w - cw_m) < 0.02)
        ece_rows.append({
            "target":          t,
            "kind":            kind,
            "n_classes":       d["n_classes"],
            "top_label_eqw":   tl_w,
            "top_label_eqm":   tl_m,
            "classwise_eqw":   cw_w,
            "classwise_eqm":   cw_m,
            "binning_stable":  stable,
        })

ece_table = pd.DataFrame(ece_rows).set_index(["target", "kind"])
print("PER-TARGET ECE TABLE")
print("=" * 80)
print("Lower is better. Compare equal-width (eqw) vs equal-mass (eqm) to")
print("check binning sensitivity. binning_stable=True means scalar ECE is")
print("trustworthy; False means rely on the reliability diagram + Brier.")
print()
print(ece_table.to_string(float_format=lambda x: f"{x:.4f}"))

ece_table_path = METRICS_DIR / "ece_table.csv"
ece_table.to_csv(ece_table_path)
print(f"\nSaved -> {ece_table_path}")


## 5. Reliability diagrams: top-label, per target

One panel per target showing the top-label reliability diagram for
the debiased probabilities. Each bar is a bin of confidence; bar height
is empirical accuracy in that bin. The diagonal $y = x$ is perfect
calibration. Bin populations are shown as a histogram below for context.

Bars **below** the diagonal = overconfident (model says it's right more
often than it is).
Bars **above** = underconfident.
The weighted gap between bars and diagonal is the ECE.


In [ ]:
def plot_reliability(ax_main, ax_hist, cal: dict, title: str = ""):
    """Plot reliability + bin-population histogram on shared x-axis."""
    bin_lo  = cal["bin_lo"]
    bin_hi  = cal["bin_hi"]
    bin_mid = 0.5 * (bin_lo + bin_hi)
    bin_acc = cal["bin_acc"]
    bin_cnt = cal["bin_count"]

    width = (bin_hi - bin_lo) * 0.9
    nonempty = bin_cnt > 0
    # Bars at empirical accuracy
    ax_main.bar(bin_mid[nonempty], bin_acc[nonempty], width=width[nonempty],
                color="#3a6f9b", edgecolor="white", linewidth=0.5,
                alpha=0.85, label="empirical accuracy")
    # Diagonal: perfect calibration
    ax_main.plot([0, 1], [0, 1], color="#c44e52", linestyle="--",
                 linewidth=1.0, label="perfect calibration")
    # Mean conf per bin as small markers, to show within-bin confidence drift
    ax_main.scatter(cal["bin_conf"][nonempty], bin_acc[nonempty],
                    color="black", s=15, zorder=5, label="bin (conf, acc)")

    ax_main.set_xlim(0, 1); ax_main.set_ylim(0, 1)
    ax_main.set_ylabel("empirical accuracy")
    ax_main.set_title(title, fontsize=10)
    ax_main.legend(loc="upper left", fontsize=7, frameon=True)
    ax_main.grid(linestyle=":", alpha=0.4)
    ax_main.spines[["top", "right"]].set_visible(False)

    # Histogram of bin populations below
    ax_hist.bar(bin_mid, bin_cnt, width=width,
                color="#888", edgecolor="white", linewidth=0.3)
    ax_hist.set_xlim(0, 1)
    ax_hist.set_xlabel("predicted top-label confidence")
    ax_hist.set_ylabel("n rows", fontsize=8)
    ax_hist.spines[["top", "right"]].set_visible(False)


# Restrict to the targets we want to display
TARGETS_TO_PLOT = ["SEX", "AGE_BINS", "EDUCATION_LEVEL", "RACE_ETHNICITY", "INCTOT_BINS", "HEALTH_INSURANCE", "ENGLISH_STATUS"]
targets_to_plot = [t for t in runs if t in TARGETS_TO_PLOT]

# Build the figure: one column per target, two stacked panels per column
n_targets = len(targets_to_plot)
ncols = min(n_targets, 4)
nrows_per_target = 2
nrows = nrows_per_target * ((n_targets + ncols - 1) // ncols)

fig = plt.figure(figsize=(4.0 * ncols,
                          1.0 + 3.5 * (nrows / nrows_per_target)))
gs = fig.add_gridspec(
    nrows=nrows, ncols=ncols,
    height_ratios=[3, 1] * (nrows // 2),
    hspace=0.65, wspace=0.3,
)

for i, t in enumerate(targets_to_plot):
    kind = "deb" if (t, "deb") in calib_data else "raw"
    cal = calib_data[(t, kind)]["top_label_eqw"]
    ece = cal["ece"]
    title = f"{t}  ({kind}, ECE={ece:.3f})"

    row_block = (i // ncols) * 2
    col       = i % ncols
    ax_main = fig.add_subplot(gs[row_block, col])
    ax_hist = fig.add_subplot(gs[row_block + 1, col], sharex=ax_main)
    plot_reliability(ax_main, ax_hist, cal, title=title)

fig.suptitle(
    "Top-label reliability diagrams (15 equal-width bins, debiased) Balanced Model",
    fontsize=12, y=.95,
)
plt.savefig(METRICS_DIR / "fig2_reliability_top_label.png",
            dpi=180, bbox_inches="tight")
plt.savefig(METRICS_DIR / "fig2_reliability_top_label.pdf",
            bbox_inches="tight")
plt.show()


## 6. Classwise reliability for the most-imbalanced target

Top-label reliability misses the per-class story when the dominant
class drives the aggregate. For RACE_ETHNICITY in particular, the
top-label diagram is essentially "is the model right when it predicts
White?" — which is yes, almost always. The interesting question is how
calibrated the model is on its predictions for *non-White* classes.

This section produces a per-class reliability diagram for one target
(default: RACE_ETHNICITY). For each class $c$ we plot
$P(Y = c \mid X)$ binned vs the empirical $\mathbb{1}[Y = c]$ rate.
Classes with very rare model predictions (most rows in low-confidence
bins) will look noisy in high-confidence bins; that's diagnostic.


In [ ]:
CLASSWISE_TARGET = "RACE_ETHNICITY"

if CLASSWISE_TARGET in runs:
    run = runs[CLASSWISE_TARGET]
    classes = run["classes"]
    kind = "deb" if (CLASSWISE_TARGET, "deb") in calib_data else "raw"
    n_classes = len(classes)
    details = calib_data[(CLASSWISE_TARGET, kind)]["classwise_eqw"]["details"]
    per_class_ece = calib_data[(CLASSWISE_TARGET, kind)]["classwise_eqw"]["per_class_ece"]

    ncols = min(n_classes, 4)
    nrows = (n_classes + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(3.5 * ncols, 3.0 * nrows),
        squeeze=False,
    )

    for c_idx, c in enumerate(classes):
        ax = axes[c_idx // ncols, c_idx % ncols]
        cal = details[c_idx]
        bin_mid = 0.5 * (cal["bin_lo"] + cal["bin_hi"])
        nonempty = cal["bin_count"] > 0

        ax.bar(bin_mid[nonempty], cal["bin_acc"][nonempty],
               width=(cal["bin_hi"] - cal["bin_lo"])[nonempty] * 0.9,
               color="#3a6f9b", edgecolor="white", linewidth=0.5,
               alpha=0.85)
        ax.plot([0, 1], [0, 1], color="#c44e52", linestyle="--",
                linewidth=1.0)
        ax.scatter(cal["bin_conf"][nonempty], cal["bin_acc"][nonempty],
                   color="black", s=12, zorder=5)

        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_title(f"{c}  (ECE={per_class_ece[c_idx]:.3f})", fontsize=10)
        ax.set_xlabel(f"P({c}|X)")
        ax.set_ylabel(f"P(Y={c})")
        ax.grid(linestyle=":", alpha=0.4)
        ax.spines[["top", "right"]].set_visible(False)

    # Hide unused subplots
    for j in range(n_classes, nrows * ncols):
        axes[j // ncols, j % ncols].set_visible(False)

    fig.suptitle(
        f"Classwise reliability for {CLASSWISE_TARGET}  "
        f"(kind={kind}, mean ECE="
        f"{calib_data[(CLASSWISE_TARGET, kind)]['classwise_eqw']['mean_ece']:.3f})",
        fontsize=12, y=1.00,
    )
    plt.tight_layout()
    plt.savefig(METRICS_DIR / f"fig3_classwise_{CLASSWISE_TARGET}.png",
                dpi=180, bbox_inches="tight")
    plt.savefig(METRICS_DIR / f"fig3_classwise_{CLASSWISE_TARGET}.pdf",
                bbox_inches="tight")
    plt.show()
else:
    print(f"{CLASSWISE_TARGET} not loaded; skipping classwise figure.")


## 7. Cross-target summary

One row per target: top-label ECE, classwise ECE, and a comparison to the
Tier 1 Brier skill score. The point is to confirm that ECE and Brier
tell consistent stories (or to flag where they diverge).


In [ ]:
# Pull Brier skill from the Tier 1 cross_target_summary if it exists, else recompute.
brier_skill = {}
cross_target_path = METRICS_DIR / "cross_target_summary.csv"
if cross_target_path.exists():
    cs = pd.read_csv(cross_target_path, index_col="target")
    for t in runs:
        if t in cs.index:
            brier_skill[t] = float(cs.loc[t, "skill_brier"])
        else:
            brier_skill[t] = np.nan
else:
    print("(Tier 1 cross_target_summary.csv not found; skill column will be NaN.)")
    for t in runs:
        brier_skill[t] = np.nan

summary_rows = []
for t in runs:
    kind = "deb" if (t, "deb") in calib_data else "raw"
    d = calib_data[(t, kind)]
    summary_rows.append({
        "target":          t,
        "n_classes":       d["n_classes"],
        "kind":            kind,
        "ece_top_label":   d["top_label_eqw"]["ece"],
        "ece_classwise":   d["classwise_eqw"]["mean_ece"],
        "ece_top_eqm":     d["top_label_eqm"]["ece"],
        "ece_cw_eqm":      d["classwise_eqm"]["mean_ece"],
        "brier_skill":     brier_skill.get(t, np.nan),
    })

ece_summary = pd.DataFrame(summary_rows).set_index("target")
print("CROSS-TARGET ECE SUMMARY  (debiased)")
print("=" * 80)
print("ece_top_label : top-label ECE, equal-width 15-bin")
print("ece_classwise : classwise mean ECE, equal-width 15-bin (sensitive to minority calibration)")
print("ece_top_eqm   : top-label ECE under equal-mass binning (binning sensitivity check)")
print("ece_cw_eqm    : classwise ECE under equal-mass binning")
print("brier_skill   : Brier skill score from Tier 1 (positive = better than uniform)")
print()
print(ece_summary.to_string(float_format=lambda x: f"{x:.4f}"))

ece_summary_path = METRICS_DIR / "ece_cross_target_summary.csv"
ece_summary.to_csv(ece_summary_path)
print(f"\nSaved -> {ece_summary_path}")
